In [ ]:
import json

import pandas as pd
import great_expectations as gx
import synapseclient

from agoradatatools.gx import GreatExpectationsRunner

context = gx.get_context(project_root_dir='../src/agoradatatools/great_expectations')

from expectations.expect_column_values_to_have_list_members_of_type import ExpectColumnValuesToHaveListMembersOfType
from expectations.expect_column_values_to_have_list_length_in_range import ExpectColumnValuesToHaveListLengthInRange

# Create Expectation Suite for RNA DE Individual Data

## Get Example Data File

In [ ]:
syn = synapseclient.Synapse()
syn.login()

In [ ]:
# To download the processed output from Synapse instead:
# rna_de_individual_file = syn.get("syn73805508").path
rna_de_individual_file = "../staging/rna_de_individual.json"

## Create Validator Object on Data File

Note: `convert_nested_columns_to_json` converts `result_order` and `data` from Python list/dict
objects into JSON strings. All expectations on these columns must use
`expect_column_values_to_match_json_schema` rather than the custom list expectations
(which expect actual Python list objects, not JSON strings).

In [ ]:
nested_columns = ["result_order", "data"]
df = pd.read_json(rna_de_individual_file)
df = GreatExpectationsRunner.convert_nested_columns_to_json(df, nested_columns)
validator = context.sources.pandas_default.read_dataframe(df)
validator.expectation_suite_name = "rna_de_individual"

## Add Expectations

In [ ]:
# ensembl_gene_id
validator.expect_column_values_to_be_of_type("ensembl_gene_id", "str")
validator.expect_column_values_to_not_be_null("ensembl_gene_id")
validator.expect_column_value_lengths_to_equal("ensembl_gene_id", 18)
validator.expect_column_values_to_match_regex("ensembl_gene_id", r"^ENSMUSG\d{11}$")

In [ ]:
# gene_symbol — can be an empty string but is never null
validator.expect_column_values_to_be_of_type("gene_symbol", "str")
validator.expect_column_values_to_not_be_null("gene_symbol")

In [ ]:
# tissue
validator.expect_column_values_to_be_of_type("tissue", "str")
validator.expect_column_values_to_not_be_null("tissue")
validator.expect_column_values_to_be_in_set(
    "tissue",
    ["Cerebral Cortex", "Hemibrain", "Hippocampus"],
)

In [ ]:
# name
validator.expect_column_values_to_be_of_type("name", "str")
validator.expect_column_values_to_not_be_null("name")

In [ ]:
# model_group
validator.expect_column_values_to_be_of_type("model_group", "str")
validator.expect_column_values_to_not_be_null("model_group")

In [ ]:
# matched_control
validator.expect_column_values_to_be_of_type("matched_control", "str")
validator.expect_column_values_to_not_be_null("matched_control")
validator.expect_column_values_to_be_in_set(
    "matched_control",
    ["B6129", "C57BL/6J"],
)

In [ ]:
# units
validator.expect_column_values_to_be_of_type("units", "str")
validator.expect_column_values_to_not_be_null("units")
validator.expect_column_values_to_be_in_set("units", ["Log2 Counts per Million"])

In [ ]:
# age
validator.expect_column_values_to_be_of_type("age", "str")
validator.expect_column_values_to_not_be_null("age")
validator.expect_column_values_to_be_in_set(
    "age",
    ["4 months", "12 months", "18 months"],
)

In [ ]:
# age_numeric
validator.expect_column_values_to_be_of_type("age_numeric", "int")
validator.expect_column_values_to_not_be_null("age_numeric")
validator.expect_column_values_to_be_in_set("age_numeric", [4, 12, 18])

In [ ]:
# result_order — JSON-encoded array of strings, length 2-4.
# Note: convert_nested_columns_to_json converts this column to a JSON string before GX
# runs, so expect_column_values_to_match_json_schema is used instead of the custom list
# expectations (which require actual Python list objects, not JSON strings).
result_order_schema = {
    "type": "array",
    "minItems": 2,
    "maxItems": 4,
    "items": {"type": "string", "minLength": 1},
}
validator.expect_column_values_to_match_json_schema(
    "result_order", json_schema=result_order_schema
)

In [ ]:
# data — JSON-encoded array of {genotype, sex, individual_id, value} objects, 19-44 items.
# Length bounds (minItems/maxItems) are enforced within the JSON schema itself.
data_schema = {
    "type": "array",
    "minItems": 19,
    "maxItems": 44,
    "items": {
        "type": "object",
        "properties": {
            "genotype":      {"type": "string", "minLength": 1},
            "sex":           {"type": "string", "minLength": 1},
            "individual_id": {"type": "string", "minLength": 1},
            "value":         {"type": "number"},
        },
        "required": ["genotype", "sex", "individual_id", "value"],
        "additionalProperties": False,
    },
}
validator.expect_column_values_to_match_json_schema("data", json_schema=data_schema)

## Save Expectation Suite and Run Checkpoint

In [ ]:
validator.save_expectation_suite(discard_failed_expectations=False)

In [ ]:
checkpoint = context.add_or_update_checkpoint(
    name="agora-test-checkpoint",
    validator=validator,
)
checkpoint_result = checkpoint.run()
context.view_validation_result(checkpoint_result)

In [ ]:
context.build_data_docs()
context.open_data_docs()